In [14]:
%%time

!python ../scripts/analyze_tuned_results.py --results-dir ../results/ours --output-dir ../results/ours --update-tex --tex-file ../docs/main.tex

HYPERPARAMETER TUNED ML MODEL RESULTS ANALYSIS
Results directory: ../results/ours
Output directory: ../results/ours

Loaded 315 result rows from new tuning scripts
Stocks: ['AAPL', 'META', 'NVDA', 'SPY', 'TSLA']
Models: ['LSTM', 'XGBoost', 'LightGBM', 'RandomForest', 'GradientBoosting', 'LogisticRegression', 'SVM']
Strategies: ['long_short']
Horizons: [2, 3, 4, 5, 6, 7, 8, 9, 10]

ANALYSIS BY MODEL

--- LSTM ---
  Samples: 45
  Avg Accuracy:    0.5066 (+/- 0.0437)
  Avg ROC-AUC:     0.5271 (+/- 0.0744)
  Avg Trades:      53.2
  Avg Win Rate:    0.4403 (44.0%)
  Avg Sharpe:      -0.6337
  Avg Total Return: 0.0345 (3.45%)

--- XGBoost ---
  Samples: 45
  Avg Accuracy:    0.4849 (+/- 0.0391)
  Avg ROC-AUC:     0.5172 (+/- 0.0350)
  Avg Trades:      53.2
  Avg Win Rate:    0.4214 (42.1%)
  Avg Sharpe:      -0.8948
  Avg Total Return: 0.0725 (7.25%)

--- LightGBM ---
  Samples: 45
  Avg Accuracy:    0.4882 (+/- 0.0374)
  Avg ROC-AUC:     0.5245 (+/- 0.0486)
  Avg Trades:      53.2
  Avg Win

In [15]:
import pandas as pd

df = pd.read_csv("../results/ours/tuned_summary_by_model_strategy.csv")
df

,Model,Strategy,Samples,Avg_Accuracy,Avg_ROC_AUC,Avg_Trades,Avg_WinRate,Avg_Sharpe,Avg_TotalReturn
0,LSTM,long_short,45,0.506586,0.527135,53.222222,0.440348,-0.633736,0.034503
1,XGBoost,long_short,45,0.484905,0.517212,53.222222,0.421388,-0.894794,0.072533
2,LightGBM,long_short,45,0.488181,0.524520,53.222222,0.435626,-0.665621,0.076949
3,RandomForest,long_short,45,0.487295,0.547106,53.222222,0.419670,-0.918532,0.055968
4,GradientBoosting,long_short,45,0.486941,0.539441,53.222222,0.427502,-0.897137,0.052945
5,LogisticRegression,long_short,45,0.494909,0.639771,53.222222,0.408835,-1.012548,0.043021
6,SVM,long_short,45,0.498451,0.615893,53.222222,0.414404,-0.990157,0.066102


In [16]:
df[df['Avg_TotalReturn'] > 0].sort_values(by='Avg_TotalReturn', ascending=False)

,Model,Strategy,Samples,Avg_Accuracy,Avg_ROC_AUC,Avg_Trades,Avg_WinRate,Avg_Sharpe,Avg_TotalReturn
2,LightGBM,long_short,45,0.488181,0.524520,53.222222,0.435626,-0.665621,0.076949
1,XGBoost,long_short,45,0.484905,0.517212,53.222222,0.421388,-0.894794,0.072533
6,SVM,long_short,45,0.498451,0.615893,53.222222,0.414404,-0.990157,0.066102
3,RandomForest,long_short,45,0.487295,0.547106,53.222222,0.419670,-0.918532,0.055968
4,GradientBoosting,long_short,45,0.486941,0.539441,53.222222,0.427502,-0.897137,0.052945
5,LogisticRegression,long_short,45,0.494909,0.639771,53.222222,0.408835,-1.012548,0.043021
0,LSTM,long_short,45,0.506586,0.527135,53.222222,0.440348,-0.633736,0.034503


In [17]:
# Per-stock Best Model Selection (for main.tex Tables 1 & 3)
# This is the unified methodology used throughout the paper

import pandas as pd

df_all = pd.read_csv('../results/ours/tuned_all_results_combined.csv')
df_ls = df_all[df_all['Strategy'] == 'long_short']

# ============================================================
# Best by AUC: For each stock, select best model by AUC
# ============================================================
print("=" * 90)
print("BEST BY AUC (Direct Per-Stock Selection)")
print("=" * 90)
print(f"{'Stock':<6} {'Model':<20} {'h':<3} {'Acc':<6} {'AUC':<6} {'N':<4} {'Win%':<6} {'Sharpe':<8} {'Ret%':<8}")
print("-" * 90)

best_auc_rows = []
for stock in ['AAPL', 'META', 'NVDA', 'SPY', 'TSLA']:
    stock_data = df_ls[df_ls['Stock'] == stock]
    if not stock_data.empty:
        best_idx = stock_data['Test_ROC_AUC'].idxmax()
        row = stock_data.loc[best_idx]
        best_auc_rows.append(row)
        model_name = 'LR' if row['Model'] == 'LogisticRegression' else row['Model']
        print(f"{stock:<6} {model_name:<20} {int(row['Horizon']):<3} {row['Test_Accuracy']:.3f} {row['Test_ROC_AUC']:.3f} {int(row['Trades']):<4} {row['WinRate']*100:.1f}  {row['Sharpe']:.2f}    {row['TotalReturn']*100:.1f}")

print("-" * 90)
best_auc_df = pd.DataFrame(best_auc_rows)
print(f"{'Avg':<6} {'':<20} {best_auc_df['Horizon'].mean():.1f} {best_auc_df['Test_Accuracy'].mean():.3f} {best_auc_df['Test_ROC_AUC'].mean():.3f} {best_auc_df['Trades'].mean():.0f}   {best_auc_df['WinRate'].mean()*100:.1f}  {best_auc_df['Sharpe'].mean():.2f}    {best_auc_df['TotalReturn'].mean()*100:.1f}")

# ============================================================
# Best by Sharpe: For each stock, select best model by Sharpe
# ============================================================
print("\n" + "=" * 90)
print("BEST BY SHARPE (Direct Per-Stock Selection)")
print("=" * 90)
print(f"{'Stock':<6} {'Model':<20} {'h':<3} {'Acc':<6} {'AUC':<6} {'N':<4} {'Win%':<6} {'Sharpe':<8} {'Ret%':<8}")
print("-" * 90)

best_sharpe_rows = []
for stock in ['AAPL', 'META', 'NVDA', 'SPY', 'TSLA']:
    stock_data = df_ls[df_ls['Stock'] == stock]
    if not stock_data.empty:
        best_idx = stock_data['Sharpe'].idxmax()
        row = stock_data.loc[best_idx]
        best_sharpe_rows.append(row)
        model_name = 'LR' if row['Model'] == 'LogisticRegression' else row['Model']
        model_name = 'GB' if row['Model'] == 'GradientBoosting' else model_name
        print(f"{stock:<6} {model_name:<20} {int(row['Horizon']):<3} {row['Test_Accuracy']:.3f} {row['Test_ROC_AUC']:.3f} {int(row['Trades']):<4} {row['WinRate']*100:.1f}  {row['Sharpe']:.2f}    {row['TotalReturn']*100:.1f}")

print("-" * 90)
best_sharpe_df = pd.DataFrame(best_sharpe_rows)
print(f"{'Avg':<6} {'':<20} {best_sharpe_df['Horizon'].mean():.1f} {best_sharpe_df['Test_Accuracy'].mean():.3f} {best_sharpe_df['Test_ROC_AUC'].mean():.3f} {best_sharpe_df['Trades'].mean():.0f}   {best_sharpe_df['WinRate'].mean()*100:.1f}  {best_sharpe_df['Sharpe'].mean():.2f}    {best_sharpe_df['TotalReturn'].mean()*100:.1f}")

print("\n" + "=" * 90)
print("SUMMARY FOR MAIN.TEX TABLE 3")
print("=" * 90)
print(f"Best by AUC:    Acc={best_auc_df['Test_Accuracy'].mean():.3f}, AUC={best_auc_df['Test_ROC_AUC'].mean():.3f}, N={best_auc_df['Trades'].mean():.0f}, Win%={best_auc_df['WinRate'].mean()*100:.1f}, Sharpe={best_auc_df['Sharpe'].mean():.2f}, Ret%={best_auc_df['TotalReturn'].mean()*100:.1f}")
print(f"Best by Sharpe: Acc={best_sharpe_df['Test_Accuracy'].mean():.3f}, AUC={best_sharpe_df['Test_ROC_AUC'].mean():.3f}, N={best_sharpe_df['Trades'].mean():.0f}, Win%={best_sharpe_df['WinRate'].mean()*100:.1f}, Sharpe={best_sharpe_df['Sharpe'].mean():.2f}, Ret%={best_sharpe_df['TotalReturn'].mean()*100:.1f}")

BEST BY AUC (Direct Per-Stock Selection)
Stock  Model                h   Acc    AUC    N    Win%   Sharpe   Ret%    
------------------------------------------------------------------------------------------
AAPL   SVM                  8   0.602 0.593 31   45.2  -1.37    -26.0
META   LR                   10  0.490 0.772 25   32.0  -1.44    -43.1
NVDA   LR                   8   0.418 0.791 31   22.6  -2.12    -66.6
SPY    SVM                  9   0.494 0.739 27   25.9  -2.05    -22.3
TSLA   LR                   5   0.566 0.606 50   56.0  1.85    157.5
------------------------------------------------------------------------------------------
Avg                         8.0 0.514 0.700 33   36.3  -1.03    -0.1

BEST BY SHARPE (Direct Per-Stock Selection)
Stock  Model                h   Acc    AUC    N    Win%   Sharpe   Ret%    
------------------------------------------------------------------------------------------
AAPL   LightGBM             9   0.530 0.508 27   74.1  1.40    29.0
MET